# NEXUS — VS Code + Jupyter + modelos locais com bootstrap automático

Este notebook foi feito **exclusivamente para execução local no VS Code/Jupyter**.

Regra desta versão:

1. o código roda localmente;
2. o Ollama roda localmente;
3. os modelos rodam localmente;
4. se algum modelo exigido pelo código não existir, ele é **baixado automaticamente**;
5. depois do bootstrap, os modelos Hugging Face são usados somente do cache local.

Não existe fallback de `qwen3:1.7b` ou `qwen3:8b` para `qwen3:4b`.
Cada identificador de modelo usado pelo código precisa existir exatamente.

## 0. Configuração local

O caminho usado nesta máquina é:

```text
/system/aulas-eb/curso_multiagente_completo/codigo/nexus
```

`REPO_ROOT` também aceita a raiz do repositório, mas nesta versão ele já fica
configurado diretamente para a pasta `codigo/nexus`.

In [1]:

from pathlib import Path
import os
import sys
import subprocess
import time
import json
import re
import uuid
import importlib
import importlib.util
import shutil
import sqlite3
import warnings

REPO_ROOT = "/system/aulas-eb/curso_multiagente_completo/codigo/nexus"

OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://127.0.0.1:11434",
).rstrip("/")

HF_HOME_LOCAL = Path(
    os.getenv(
        "HF_HOME",
        str(Path.home() / ".cache" / "huggingface"),
    )
).expanduser()

HF_HUB_LOCAL = Path(
    os.getenv(
        "HF_HUB_CACHE",
        str(HF_HOME_LOCAL / "hub"),
    )
).expanduser()

U2NET_HOME_LOCAL = Path(
    os.getenv(
        "U2NET_HOME",
        str(Path.home() / ".u2net"),
    )
).expanduser()

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL

AUTO_START_OLLAMA = True
AUTO_INSTALL_DEPENDENCIAS_AUSENTES = True
AUTO_DOWNLOAD_MODELOS_AUSENTES = True

HF_HOME_LOCAL.mkdir(parents=True, exist_ok=True)
HF_HUB_LOCAL.mkdir(parents=True, exist_ok=True)
U2NET_HOME_LOCAL.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Kernel:", sys.executable)
print("CWD:", Path.cwd())


Python: 3.11.16 (main, Sep  1 2026, 14:18:37) [Clang 22.1.3 ]
Kernel: /system/aulas-eb/.venv/bin/python
CWD: /system/aulas-eb


In [2]:

def localizar_repo():
    """
    Aceita:
      1) caminho direto para .../codigo/nexus
      2) raiz que contém codigo/nexus
    """
    if REPO_ROOT:
        p = Path(REPO_ROOT).expanduser().resolve()

        if p.is_dir() and p.name == "nexus" and p.parent.name == "codigo":
            return p.parent.parent.resolve(), p

        nexus = p / "codigo" / "nexus"
        if nexus.is_dir():
            return p.resolve(), nexus.resolve()

        raise FileNotFoundError(
            f"Caminho inválido: {p}\n"
            "Informe diretamente .../codigo/nexus "
            "ou a raiz que contém codigo/nexus."
        )

    for p in [Path.cwd(), *Path.cwd().parents]:
        if p.name == "nexus" and p.parent.name == "codigo":
            return p.parent.parent.resolve(), p.resolve()

        nexus = p / "codigo" / "nexus"
        if nexus.is_dir():
            return p.resolve(), nexus.resolve()

    raise FileNotFoundError(
        "Repositório local não encontrado."
    )

R, N = localizar_repo()
os.chdir(N)

print("Repositório:", R)
print("NEXUS:", N)


Repositório: /system/aulas-eb/curso_multiagente_completo
NEXUS: /system/aulas-eb/curso_multiagente_completo/codigo/nexus


## 1. Validar e completar o ambiente Python

A checagem usa `find_spec()` e não importa bibliotecas pesadas durante o diagnóstico.

Se alguma dependência do curso estiver ausente, o notebook usa o
`requirements.txt` do próprio projeto. Para o demo `space_demo`, usa o
`requirements.txt` específico dele.

In [3]:

def modulo_existe(nome: str) -> bool:
    try:
        return importlib.util.find_spec(nome) is not None
    except (ImportError, ModuleNotFoundError, AttributeError):
        return False

MODULOS_PRINCIPAIS = {
    "requests": "requests",
    "pytest": "pytest",
    "openai": "openai",
    "pydantic": "pydantic",
    "langchain": "langchain",
    "langchain_core": "langchain-core",
    "langchain_community": "langchain-community",
    "langchain_ollama": "langchain-ollama",
    "langchain_chroma": "langchain-chroma",
    "langchain_huggingface": "langchain-huggingface",
    "langchain_text_splitters": "langchain-text-splitters",
    "langgraph": "langgraph",
    "langgraph.checkpoint.sqlite": "langgraph-checkpoint-sqlite",
    "chromadb": "chromadb",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers",
    "huggingface_hub": "huggingface_hub",
    "torch": "torch",
    "pandas": "pandas",
    "wordcloud": "wordcloud",
    "matplotlib": "matplotlib",
}

def ausentes(mapa):
    return {
        modulo: pacote
        for modulo, pacote in mapa.items()
        if not modulo_existe(modulo)
    }

faltando = ausentes(MODULOS_PRINCIPAIS)

if faltando and AUTO_INSTALL_DEPENDENCIAS_AUSENTES:
    print("Dependências principais ausentes:")
    for modulo, pacote in faltando.items():
        print(" -", modulo, "->", pacote)

    req = N / "requirements.txt"
    if not req.exists():
        raise FileNotFoundError(req)

    print("\nInstalando requirements.txt no kernel atual...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(req),
        ],
        check=True,
    )
    importlib.invalidate_caches()

faltando = ausentes(MODULOS_PRINCIPAIS)
if faltando:
    raise RuntimeError(
        "Dependências principais continuam ausentes:\n"
        + "\n".join(
            f"- {m} ({p})"
            for m, p in faltando.items()
        )
    )

MODULOS_SPACE_DEMO = {
    "rembg": "rembg",
    "PIL": "pillow",
    "onnxruntime": "onnxruntime",
}

faltando_space = ausentes(MODULOS_SPACE_DEMO)

if faltando_space and AUTO_INSTALL_DEPENDENCIAS_AUSENTES:
    print("\nDependências do dia5/space_demo ausentes:")
    for modulo, pacote in faltando_space.items():
        print(" -", modulo, "->", pacote)

    req_space = N / "dia5" / "space_demo" / "requirements.txt"
    if not req_space.exists():
        raise FileNotFoundError(req_space)

    print("\nInstalando requirements do space_demo...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(req_space),
        ],
        check=True,
    )
    importlib.invalidate_caches()

faltando_space = ausentes(MODULOS_SPACE_DEMO)
if faltando_space:
    raise RuntimeError(
        "Dependências do space_demo continuam ausentes:\n"
        + "\n".join(
            f"- {m} ({p})"
            for m, p in faltando_space.items()
        )
    )

print("\n[OK] Ambiente Python completo.")



[OK] Ambiente Python completo.


## 2. Manifesto exato dos modelos usados pelo código

O manifesto abaixo foi conferido contra os arquivos Python do projeto.
Não há substituição automática por outro tamanho de modelo.

In [4]:

OLLAMA = {
    "small": "qwen3:1.7b",
    "base": "qwen3:4b",
    "medium": "qwen3:4b",
    "large": "qwen3:8b",
    "embedding": "nomic-embed-text",
}

HF = {
    "embedding": "intfloat/multilingual-e5-small",
    "reranker": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "sentiment": "nlptown/bert-base-multilingual-uncased-sentiment",
    "zero_shot": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    "qa": "deepset/xlm-roberta-base-squad2",
    "chat": "Qwen/Qwen3-0.6B",
    "vllm": "Qwen/Qwen3-8B",
}

REMBG_MODEL = "u2net"

OLLAMA_OBRIGATORIOS = sorted(set(OLLAMA.values()))
HF_OBRIGATORIOS = sorted(set(HF.values()))

os.environ["MODELO"] = OLLAMA["base"]
os.environ["MODELO_PEQUENO"] = OLLAMA["small"]
os.environ["MODELO_MEDIO"] = OLLAMA["medium"]
os.environ["MODELO_GRANDE"] = OLLAMA["large"]
os.environ["VLLM_MODELO"] = HF["vllm"]

os.environ["HF_HOME"] = str(HF_HOME_LOCAL)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_LOCAL)
os.environ["U2NET_HOME"] = str(U2NET_HOME_LOCAL)

for chave in (
    "HF_HUB_OFFLINE",
    "TRANSFORMERS_OFFLINE",
    "HF_DATASETS_OFFLINE",
):
    os.environ.pop(chave, None)

print("Ollama obrigatórios:")
for m in OLLAMA_OBRIGATORIOS:
    print(" -", m)

print("\nHugging Face obrigatórios:")
for m in HF_OBRIGATORIOS:
    print(" -", m)

print("\nrembg:")
print(" -", REMBG_MODEL)


Ollama obrigatórios:
 - nomic-embed-text
 - qwen3:1.7b
 - qwen3:4b
 - qwen3:8b

Hugging Face obrigatórios:
 - MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
 - Qwen/Qwen3-0.6B
 - Qwen/Qwen3-8B
 - cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
 - deepset/xlm-roberta-base-squad2
 - intfloat/multilingual-e5-small
 - nlptown/bert-base-multilingual-uncased-sentiment

rembg:
 - u2net


## 2.1 Auditoria automática dos identificadores de modelo no código

Esta célula percorre todos os arquivos `.py` do projeto. Se aparecer um
identificador de modelo que não esteja no manifesto, a execução é interrompida.

In [5]:

arquivos_py = [
    p
    for p in N.rglob("*.py")
    if "__pycache__" not in p.parts
]

codigo_total = "\n".join(
    p.read_text(
        encoding="utf-8",
        errors="ignore",
    )
    for p in arquivos_py
)

ollama_encontrados = {
    m.lower()
    for m in re.findall(
        r"\b(?:qwen3:[0-9.]+b|nomic-embed-text)\b",
        codigo_total,
        flags=re.I,
    )
}

familias_hf = (
    "Qwen",
    "intfloat",
    "cross-encoder",
    "nlptown",
    "MoritzLaurer",
    "deepset",
)

hf_regex = re.compile(
    r"[\"']((?:"
    + "|".join(re.escape(x) for x in familias_hf)
    + r")/[A-Za-z0-9_.-]+)[\"']"
)

hf_encontrados = set(
    hf_regex.findall(codigo_total)
)

ollama_manifesto = {
    x.lower()
    for x in OLLAMA_OBRIGATORIOS
}
hf_manifesto = set(HF_OBRIGATORIOS)

nao_manifestados_ollama = (
    ollama_encontrados
    - ollama_manifesto
)
nao_manifestados_hf = (
    hf_encontrados
    - hf_manifesto
)

print("Ollama encontrados no código:")
for x in sorted(ollama_encontrados):
    print(" -", x)

print("\nHF encontrados no código:")
for x in sorted(hf_encontrados):
    print(" -", x)

if nao_manifestados_ollama or nao_manifestados_hf:
    raise RuntimeError(
        "Há modelos usados pelo código que não estão no manifesto.\n"
        f"Ollama: {sorted(nao_manifestados_ollama)}\n"
        f"HF: {sorted(nao_manifestados_hf)}"
    )

print("\n[OK] Todos os modelos explícitos do código estão no manifesto.")


Ollama encontrados no código:
 - nomic-embed-text
 - qwen3:1.7b
 - qwen3:4b
 - qwen3:8b

HF encontrados no código:
 - MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
 - Qwen/Qwen3-0.6B
 - Qwen/Qwen3-8B
 - cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
 - deepset/xlm-roberta-base-squad2
 - intfloat/multilingual-e5-small
 - nlptown/bert-base-multilingual-uncased-sentiment

[OK] Todos os modelos explícitos do código estão no manifesto.


## 3. Ollama local

O serviço local é iniciado automaticamente se não estiver respondendo.

In [6]:

import requests

if not shutil.which("ollama"):
    raise RuntimeError(
        "Executável 'ollama' não encontrado no PATH do VS Code."
    )

def ollama_ativo():
    try:
        return requests.get(
            f"{OLLAMA_BASE_URL}/api/tags",
            timeout=2,
        ).ok
    except Exception:
        return False

OLLAMA_PROC = None

if not ollama_ativo() and AUTO_START_OLLAMA:
    print("Ollama não respondeu; iniciando 'ollama serve'...")
    OLLAMA_PROC = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    for _ in range(30):
        if ollama_ativo():
            break
        time.sleep(1)

if not ollama_ativo():
    raise RuntimeError(
        f"Ollama não responde em {OLLAMA_BASE_URL}."
    )

print("[OK] Ollama:", OLLAMA_BASE_URL)


[OK] Ollama: http://127.0.0.1:11434


## 4. Garantir todos os modelos Ollama

Modelo ausente é baixado com `ollama pull`. A execução só continua
depois que todos os identificadores exigidos existem localmente.

In [7]:

def canon(nome):
    n = (nome or "").strip().lower()
    return n[:-7] if n.endswith(":latest") else n

def modelos_ollama_instalados():
    resp = requests.get(
        f"{OLLAMA_BASE_URL}/api/tags",
        timeout=30,
    )
    resp.raise_for_status()

    raw = {
        m.get("name", "")
        for m in resp.json().get("models", [])
    }
    return raw, {canon(x) for x in raw}

raw, presentes = modelos_ollama_instalados()

print("Antes do bootstrap:")
for nome in sorted(raw):
    print(" -", nome)

for modelo in OLLAMA_OBRIGATORIOS:
    if canon(modelo) in presentes:
        print("[OK]", modelo)
        continue

    if not AUTO_DOWNLOAD_MODELOS_AUSENTES:
        raise RuntimeError(
            f"Modelo Ollama ausente: {modelo}"
        )

    print(f"\n[DOWNLOAD OLLAMA] {modelo}")

    resultado = subprocess.run(
        ["ollama", "pull", modelo],
        check=False,
    )

    if resultado.returncode != 0:
        raise RuntimeError(
            f"Falha ao baixar o modelo Ollama {modelo} "
            f"(código {resultado.returncode})."
        )

    raw, presentes = modelos_ollama_instalados()

faltando = [
    modelo
    for modelo in OLLAMA_OBRIGATORIOS
    if canon(modelo) not in presentes
]

if faltando:
    raise RuntimeError(
        "Após o bootstrap ainda faltam modelos Ollama:\n"
        + "\n".join(f"- {x}" for x in faltando)
    )

print("\n[OK] Todos os modelos Ollama exigidos pelo código existem.")


Antes do bootstrap:
 - nomic-embed-text:latest
 - qwen3:1.7b
 - qwen3:4b
 - qwen3:8b
[OK] nomic-embed-text
[OK] qwen3:1.7b
[OK] qwen3:4b
[OK] qwen3:8b

[OK] Todos os modelos Ollama exigidos pelo código existem.


In [8]:

# Usa DIRETAMENTE o daemon Ollama já ouvindo em OLLAMA_BASE_URL.
# Nada de "ollama run": a inferência é feita por HTTP no processo existente.

print("Daemon Ollama:", OLLAMA_BASE_URL)

try:
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": OLLAMA["base"],
            "prompt": "Responda apenas OK.",
            "stream": False,
            "think": False,
            "keep_alive": "5m",
            "options": {
                "temperature": 0,
                "num_predict": 8,
            },
        },
        timeout=(5, 120),
    )
    resp.raise_for_status()
except requests.exceptions.Timeout as e:
    raise RuntimeError(
        f"O daemon {OLLAMA_BASE_URL} está acessível, "
        "mas a inferência excedeu 120 segundos."
    ) from e
except requests.exceptions.RequestException as e:
    raise RuntimeError(
        f"Falha ao chamar o daemon Ollama em {OLLAMA_BASE_URL}: {e}"
    ) from e

resultado = resp.json()

print("Resposta:", (resultado.get("response") or "").strip())
print("Modelo:", resultado.get("model"))
print("Concluído:", resultado.get("done"))

if resultado.get("done") is not True:
    raise RuntimeError(
        f"Geração não concluída: {resultado}"
    )

# Consulta o mesmo processo pela API oficial.
try:
    resp_ps = requests.get(
        f"{OLLAMA_BASE_URL}/api/ps",
        timeout=10,
    )
    resp_ps.raise_for_status()
except requests.exceptions.RequestException as e:
    raise RuntimeError(
        f"Falha ao consultar /api/ps em {OLLAMA_BASE_URL}: {e}"
    ) from e

ps = resp_ps.json()

print("\nModelos carregados:")
for modelo in ps.get("models", []):
    print(
        " -",
        modelo.get("name") or modelo.get("model"),
        "| size_vram:",
        modelo.get("size_vram"),
    )

if not ps.get("models"):
    print(" - nenhum modelo carregado")

if shutil.which("nvidia-smi"):
    print("\nGPU:")
    subprocess.run(
        ["nvidia-smi"],
        check=False,
    )


Daemon Ollama: http://127.0.0.1:11434
Resposta: Okay, the user wants me to respond
Modelo: qwen3:4b
Concluído: True

Modelos carregados:
 - qwen3:4b | size_vram: 3178149969

GPU:
Wed Sep  9 00:12:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...    Off |   00000000:02:00.0 Off |                  N/A |
| N/A   64C    P8             91W /   80W |    3181Mi

## 5. Garantir todos os modelos Hugging Face

Cada snapshot é procurado no cache. O que estiver ausente é baixado.
Após a validação, o restante do notebook usa apenas os snapshots locais.

In [9]:

from huggingface_hub import snapshot_download

def resolver_hf_local(repo_id: str):
    try:
        return snapshot_download(
            repo_id=repo_id,
            cache_dir=str(HF_HUB_LOCAL),
            local_files_only=True,
        )
    except Exception:
        return None

HF_LOCAL = {}

for chave, repo_id in HF.items():
    local = resolver_hf_local(repo_id)

    if local:
        HF_LOCAL[chave] = local
        print("[OK HF]", repo_id)
        continue

    if not AUTO_DOWNLOAD_MODELOS_AUSENTES:
        raise RuntimeError(
            f"Modelo Hugging Face ausente: {repo_id}"
        )

    print(f"\n[DOWNLOAD HF] {repo_id}")

    for flag in (
        "HF_HUB_OFFLINE",
        "TRANSFORMERS_OFFLINE",
        "HF_DATASETS_OFFLINE",
    ):
        os.environ.pop(flag, None)

    local = snapshot_download(
        repo_id=repo_id,
        cache_dir=str(HF_HUB_LOCAL),
        local_files_only=False,
    )

    HF_LOCAL[chave] = local

hf_faltando = []

for chave, repo_id in HF.items():
    local = resolver_hf_local(repo_id)

    if not local:
        hf_faltando.append(repo_id)
    else:
        HF_LOCAL[chave] = local

if hf_faltando:
    raise RuntimeError(
        "Após o bootstrap ainda faltam modelos Hugging Face:\n"
        + "\n".join(f"- {x}" for x in hf_faltando)
    )

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("\n[OK] Todos os modelos Hugging Face exigidos pelo código existem.")
for chave, caminho in HF_LOCAL.items():
    print(f" - {chave}: {caminho}")


[OK HF] intfloat/multilingual-e5-small
[OK HF] cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
[OK HF] nlptown/bert-base-multilingual-uncased-sentiment
[OK HF] MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
[OK HF] deepset/xlm-roberta-base-squad2
[OK HF] Qwen/Qwen3-0.6B
[OK HF] Qwen/Qwen3-8B

[OK] Todos os modelos Hugging Face exigidos pelo código existem.
 - embedding: /home/over/.cache/huggingface/hub/models--intfloat--multilingual-e5-small/snapshots/614241f622f53c4eeff9890bdc4f31cfecc418b3
 - reranker: /home/over/.cache/huggingface/hub/models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1/snapshots/1427fd652930e4ba29e8149678df786c240d8825
 - sentiment: /home/over/.cache/huggingface/hub/models--nlptown--bert-base-multilingual-uncased-sentiment/snapshots/8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d
 - zero_shot: /home/over/.cache/huggingface/hub/models--MoritzLaurer--mDeBERTa-v3-base-mnli-xnli/snapshots/8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c
 - qa: /home/over/.cache/huggingface/hub/models--deepset--x

/system/aulas-eb/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5.1 Garantir `u2net` para o `rembg`

O `space_demo` usa `rembg.remove()`. Esta etapa garante previamente o
peso `u2net`, para que o demo também tenha o modelo local.

In [10]:

from pathlib import Path
import hashlib

from rembg import new_session
from rembg.sessions.u2net import U2netSession

U2NET_MD5_OFICIAL = "60024c5c889badc19c04ad937298a77b"

def md5_arquivo(caminho: Path, bloco: int = 1024 * 1024) -> str:
    h = hashlib.md5()
    with caminho.open("rb") as f:
        while True:
            chunk = f.read(bloco)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# O próprio rembg resolve:
# - layout atual: ~/.rembg/models/u2net/u2net.onnx
# - layout legado: ~/.u2net/u2net.onnx
#
# Se o modelo não existir, download_models() baixa o arquivo oficial
# e RETORNA o caminho real utilizado pela versão instalada.
print("Resolvendo modelo rembg:", REMBG_MODEL)

try:
    caminho_modelo = Path(
        U2netSession.download_models()
    ).expanduser().resolve()
except Exception as e:
    raise RuntimeError(
        "Falha ao localizar/baixar o modelo u2net pelo próprio rembg."
    ) from e

print("Modelo resolvido em:", caminho_modelo)

if not caminho_modelo.is_file():
    raise RuntimeError(
        "O rembg informou um caminho de modelo que não existe: "
        + str(caminho_modelo)
    )

tamanho = caminho_modelo.stat().st_size
if tamanho <= 0:
    raise RuntimeError(
        "Arquivo u2net vazio: "
        + str(caminho_modelo)
    )

md5_atual = md5_arquivo(caminho_modelo)

print("Tamanho:", tamanho, "bytes")
print("MD5:", md5_atual)

if md5_atual.lower() != U2NET_MD5_OFICIAL:
    raise RuntimeError(
        "Checksum inválido para u2net.\n"
        f"Arquivo: {caminho_modelo}\n"
        f"Esperado: {U2NET_MD5_OFICIAL}\n"
        f"Obtido:   {md5_atual}"
    )

# Guardar o caminho REAL para as células seguintes.
REMBG_MODEL_PATH = caminho_modelo

# Só agora cria a sessão ONNX.
try:
    sessao_bootstrap = new_session(REMBG_MODEL)
finally:
    if "sessao_bootstrap" in globals():
        del sessao_bootstrap

print("[OK] Modelo rembg/u2net validado:", REMBG_MODEL_PATH)


Resolvendo modelo rembg: u2net
Modelo resolvido em: /home/over/.u2net/models/u2net/u2net.onnx
Tamanho: 175997641 bytes
MD5: 60024c5c889badc19c04ad937298a77b
[OK] Modelo rembg/u2net validado: /home/over/.u2net/models/u2net/u2net.onnx


## 6. Isolar os módulos de cada dia

In [11]:
MODULOS = {
    "agente",
    "clientes",
    "ferramentas",
    "indexar",
    "nexus",
    "hooks",
    "steering",
    "ferramentas_web",
    "equipe",
    "prompts",
    "handoff",
    "memoria_semantica",
    "email_assistente",
    "app_gradio",
    "avaliacao",
    "modelos",
    "roteador",
    "pipelines",
    "embeddings_hf",
    "cli",
}

def usar_dia(nome):
    pasta = N / nome
    if not pasta.exists():
        raise FileNotFoundError(pasta)

    for mod in list(sys.modules):
        if mod in MODULOS:
            sys.modules.pop(mod, None)

    caminhos = [
        str(N / f"dia{i}")
        for i in range(1, 6)
    ]
    sys.path[:] = [
        p
        for p in sys.path
        if p not in caminhos
    ]
    sys.path.insert(0, str(pasta))
    importlib.invalidate_caches()
    os.chdir(N)
    print("[DIA]", nome)

## 7. Compilação e testes

In [12]:
comp = subprocess.run(
    [
        sys.executable,
        "-m",
        "compileall",
        "-q",
        str(N),
    ],
    text=True,
    capture_output=True,
    check=False,
)

if comp.returncode:
    print(comp.stdout)
    print(comp.stderr)
    raise RuntimeError("Falha de compilação.")

print("[OK] compileall")

tests = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        str(N / "testes"),
    ],
    cwd=str(N),
    text=True,
    capture_output=True,
    check=False,
)

print(tests.stdout)
if tests.stderr:
    print(tests.stderr)

if tests.returncode:
    raise RuntimeError("Há testes falhando.")

print("[OK] testes.")

[OK] compileall
.......................................................                  [100%]
55 passed in 0.26s

[OK] testes.


# DIA 1 — Python puro + Ollama + tools

In [13]:
usar_dia("dia1")
import ferramentas as f1

print("950 * 24 =", f1.calcular("950 * 24"))
print(
    "Traversal:",
    f1.ler_arquivo("../../etc/passwd"),
)

[DIA] dia1
950 * 24 = 22800
Traversal: ERRO: caminho fora da pasta permitida. Use apenas arquivos existentes em docs/.


In [14]:
usar_dia("dia1")
import agente as agente1

resposta_d1 = agente1.rodar(
    "Qual foi o faturamento de 2024? "
    "Leia o documento necessário e cite o nome dele.",
    backend="ollama",
    max_passos=8,
    verbose=True,
)

print(resposta_d1)

[DIA] dia1
[passo 0] listar_arquivos({}) -> 2023_relatorio.md 2024_relatorio.md estatuto.md fornecedores.md notas_reuniao.md | 1189 tokens
[passo 1] ler_arquivo({'caminho': '2024_relatorio.md'}) -> # Relatório Anual 2024 — Cooperativa Vale Verde  ## Resumo executivo  Faturamento bruto de R$ 5.640.000,00 em 2024. O nú | 3150 tokens
[passo 2] resposta final | 5707 tokens | 23.3s
O faturamento de 2024 foi R$ 5.640.000,00. O documento utilizado foi **2024_relatorio.md**.


# DIA 2 — LangChain + RAG/Chroma

In [15]:
usar_dia("dia2")
import indexar as indexar2

banco_d2 = indexar2.construir(
    recriar=not (N / ".chroma").exists()
)

for doc in banco_d2.similarity_search(
    "faturamento de 2024",
    k=2,
):
    print(
        "\n---",
        Path(doc.metadata.get("source", "?")).name,
    )
    print(doc.page_content[:500])

[DIA] dia2
5 documentos -> 7 trechos

--- 2024_relatorio.md
# Relatório Anual 2024 — Cooperativa Vale Verde

## Resumo executivo

Faturamento bruto de R$ 5.640.000,00 em 2024. O número de cooperados chegou a 261.
A linha de café especial passou a responder por mais da metade da receita.

## Faturamento por linha

| Linha | Faturamento (R$) | Participação |
|---|---|---|
| Café especial | 3.102.000 | 55% |
| Hortifrúti | 1.410.000 | 25% |
| Laticínios | 1.128.000 | 20% |

## Custos

Custo operacional total: R$ 4.230.000,00. Margem operacional de 25%.
Logí

--- 2023_relatorio.md
# Relatório Anual 2023 — Cooperativa Vale Verde

## Resumo executivo

O exercício de 2023 encerrou com faturamento bruto de R$ 4.820.000,00, crescimento de
12% sobre 2022. O número de cooperados subiu de 214 para 238.

## Faturamento por linha

| Linha | Faturamento (R$) | Participação |
|---|---|---|
| Café especial | 2.410.000 | 50% |
| Hortifrúti | 1.446.000 | 30% |
| Laticínios | 964.000 | 20% |

## Custos

C

In [16]:
usar_dia("dia2")
import agente as agente2

print(
    agente2.rodar(
        "Qual foi o faturamento de 2024? Cite a fonte.",
        max_passos=8,
        verbose=True,
    )
)

[DIA] dia2
[passo 0] buscar_documentos({'consulta': 'faturamento 2024', 'k': 1}) -> [fonte: 2024_relatorio.md]
# Relatório Anual 2024 — Cooperativa Vale Verde

## R
O faturamento de 2024 foi de R$ 5.640.000,00 [fonte: 2024_relatorio.md].


# DIA 3 — LangGraph + SQLite local

In [17]:
usar_dia("dia3")
import hooks
import steering

print(
    hooks.avaliar_politicas(
        "ler_arquivo",
        {"caminho": "../../etc/passwd"},
        {},
    )
)
print(
    steering.detectar_estagnacao(
        {"passos": 7, "achados": []}
    )
)

[DIA] dia3
leitura fora de docs/ nao e permitida
True


In [18]:
usar_dia("dia3")

from langchain_core.messages import HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver
import nexus as nexus3

conn_d3 = sqlite3.connect(
    str(N / "nexus_vscode_offline.db"),
    check_same_thread=False,
)

try:
    saver_d3 = SqliteSaver(conn_d3)

    grafo_d3 = nexus3.construir_grafo().compile(
        checkpointer=saver_d3,
        interrupt_before=[],
    )

    estado_d3 = grafo_d3.invoke(
        {
            "messages": [
                HumanMessage(
                    "Qual foi o faturamento de 2024? Cite a fonte."
                )
            ],
            "passos": 0,
        },
        {
            "configurable": {
                "thread_id": "vscode-offline-dia3"
            },
            "recursion_limit": 30,
        },
    )

    print(estado_d3["messages"][-1].content)
finally:
    conn_d3.close()

[DIA] dia3
O faturamento de 2024 foi de R$ 5.640.000,00 [fonte: 2024_relatorio.md].


# DIA 4 — Equipe multiagente + SQLite local

In [19]:
usar_dia("dia4")

from langchain_core.messages import HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver
import equipe as equipe4

conn_d4 = sqlite3.connect(
    str(N / "equipe_vscode_offline.db"),
    check_same_thread=False,
)

try:
    saver_d4 = SqliteSaver(conn_d4)
    grafo_d4 = equipe4.construir_equipe().compile(
        checkpointer=saver_d4
    )

    pergunta = (
        "Compare os fornecedores disponíveis "
        "e recomende um, citando fontes."
    )

    entrada = {
        "messages": [HumanMessage(pergunta)],
        "pergunta": pergunta,
        "proximo": "",
        "instrucao": pergunta,
        "achados": [],
        "rascunho": "",
        "veredito": "",
        "rodadas": 0,
    }

    final_d4 = grafo_d4.invoke(
        entrada,
        {
            "configurable": {
                "thread_id": "vscode-offline-dia4"
            },
            "recursion_limit": 40,
        },
    )

    print("VEREDITO:", final_d4.get("veredito"))
    print(
        final_d4.get("rascunho")
        or final_d4["messages"][-1].content
    )
finally:
    conn_d4.close()

[DIA] dia4


KeyboardInterrupt: 

## Dia 4 — Assistente de e-mail local

In [ ]:
usar_dia("dia2")
import indexar as idx_email

banco_email = idx_email.construir(
    recriar=False
)

usar_dia("dia4")
import email_assistente as email4

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command

conn_email = sqlite3.connect(
    str(N / "email_vscode_offline.db"),
    check_same_thread=False,
)

try:
    saver_email = SqliteSaver(conn_email)

    grafo_email = email4.construir_grafo(
        retriever=banco_email
    )
    grafo_email.checkpointer = saver_email

    emails = json.loads(
        (N / "dados" / "emails.json").read_text(
            encoding="utf-8"
        )
    )

    cfg = {
        "configurable": {
            "thread_id": "vscode-offline-email-0"
        }
    }

    estado_email = grafo_email.invoke(
        emails[0],
        cfg,
    )

    snapshot = grafo_email.get_state(cfg)

    if snapshot.tasks and snapshot.tasks[0].interrupts:
        print(
            snapshot.tasks[0]
            .interrupts[0]
            .value
        )

        estado_email = grafo_email.invoke(
            Command(
                resume={"acao": "descartar"}
            ),
            cfg,
        )

    print(estado_email)
finally:
    conn_email.close()

# DIA 5 — modelos locais Hugging Face + Ollama

In [ ]:
import torch

HF_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
HF_PIPELINE_DEVICE = (
    0
    if torch.cuda.is_available()
    else -1
)

print("torch:", torch.__version__)
print("device:", HF_DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

usar_dia("dia5")
import modelos as modelos5

esperado = {
    "supervisor": OLLAMA["small"],
    "pesquisador": OLLAMA["medium"],
    "analista": OLLAMA["small"],
    "redator": OLLAMA["large"],
    "critico": OLLAMA["medium"],
}

for papel, modelo in esperado.items():
    atual = modelos5.para(papel).model
    print(papel, "->", atual)
    assert atual == modelo

## Dia 5.1 — Embeddings locais

In [ ]:
if not HF_LOCAL["embedding"]:
    raise RuntimeError(
        "Snapshot local ausente: "
        + HF["embedding"]
    )

from langchain_huggingface import HuggingFaceEmbeddings

emb_hf = HuggingFaceEmbeddings(
    model_name=HF_LOCAL["embedding"],
    model_kwargs={
        "device": HF_DEVICE,
    },
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

print(
    "dim:",
    len(
        emb_hf.embed_query(
            "faturamento de 2024"
        )
    ),
)

## Dia 5.2 — Chroma HF local

In [ ]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
)
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
)

PERSIST_HF = N / ".chroma_hf_offline"

if PERSIST_HF.exists():
    banco_hf = Chroma(
        persist_directory=str(PERSIST_HF),
        embedding_function=emb_hf,
    )
else:
    documentos = DirectoryLoader(
        str(N / "docs"),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    ).load()

    partes = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120,
    ).split_documents(documentos)

    banco_hf = Chroma.from_documents(
        partes,
        emb_hf,
        persist_directory=str(PERSIST_HF),
    )

print("[OK]", PERSIST_HF)

## Dia 5.3 — Reranker local

In [ ]:
if not HF_LOCAL["reranker"]:
    raise RuntimeError(
        "Snapshot local ausente: "
        + HF["reranker"]
    )

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    HF_LOCAL["reranker"],
    device=HF_DEVICE,
)

consulta = "custo total dos fornecedores em 24 meses"
candidatos = banco_hf.similarity_search(
    consulta,
    k=10,
)

scores = reranker.predict(
    [
        (consulta, doc.page_content)
        for doc in candidatos
    ]
)

ranking = sorted(
    zip(scores, candidatos),
    key=lambda x: -float(x[0]),
)

for score, doc in ranking[:4]:
    print("\nscore:", float(score))
    print(
        Path(
            doc.metadata.get("source", "?")
        ).name
    )
    print(doc.page_content[:350])

## Dia 5.4 — Sentimento e zero-shot locais

In [ ]:
from transformers import pipeline

if not HF_LOCAL["sentiment"]:
    raise RuntimeError(
        "Snapshot local ausente: "
        + HF["sentiment"]
    )

sentimento = pipeline(
    "sentiment-analysis",
    model=HF_LOCAL["sentiment"],
    tokenizer=HF_LOCAL["sentiment"],
    device=HF_PIPELINE_DEVICE,
)

print(
    sentimento(
        [
            "Excelente equipamento.",
            "Atendimento péssimo.",
        ],
        truncation=True,
    )
)

if not HF_LOCAL["zero_shot"]:
    raise RuntimeError(
        "Snapshot local ausente: "
        + HF["zero_shot"]
    )

usar_dia("dia5")
import pipelines as p5

zero_shot = pipeline(
    "zero-shot-classification",
    model=HF_LOCAL["zero_shot"],
    tokenizer=HF_LOCAL["zero_shot"],
    device=HF_PIPELINE_DEVICE,
)

print(
    zero_shot(
        "Preciso comprar um torno industrial.",
        candidate_labels=p5.CATEGORIAS,
    )
)

## Dia 5.5 — QA extrativo local

In [ ]:
if not HF_LOCAL["qa"]:
    raise RuntimeError(
        "Snapshot local ausente: "
        + HF["qa"]
    )

from transformers import pipeline

qa = pipeline(
    "question-answering",
    model=HF_LOCAL["qa"],
    tokenizer=HF_LOCAL["qa"],
    device=HF_PIPELINE_DEVICE,
)

contexto = (
    N
    / "dados"
    / "faq"
    / "politicas.md"
).read_text(encoding="utf-8")

print(
    qa(
        question="Qual é a política descrita no documento?",
        context=contexto,
    )
)

## Dia 5.6 — Qwen3-0.6B local

In [ ]:

if not HF_LOCAL["chat"]:
    raise RuntimeError(
        "Snapshot local ausente: "
        + HF["chat"]
    )

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

chat_path = HF_LOCAL["chat"]

tok = AutoTokenizer.from_pretrained(
    chat_path,
    local_files_only=True,
)

dtype_chat = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

modelo_chat = AutoModelForCausalLM.from_pretrained(
    chat_path,
    local_files_only=True,
    torch_dtype=dtype_chat,
)

modelo_chat = modelo_chat.to(HF_DEVICE)
modelo_chat.eval()

mensagens = [
    {
        "role": "system",
        "content": "Responda em português e objetivamente.",
    },
    {
        "role": "user",
        "content": (
            "Explique em duas frases o papel "
            "de um supervisor multiagente."
        ),
    },
]

inputs = tok.apply_chat_template(
    mensagens,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
)

inputs = {
    k: v.to(HF_DEVICE)
    for k, v in inputs.items()
}

with torch.inference_mode():
    saida = modelo_chat.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )

n = inputs["input_ids"].shape[-1]
texto = tok.decode(
    saida[0][n:],
    skip_special_tokens=True,
).strip()

if "<think>" in texto:
    raise AssertionError(
        "O modelo emitiu bloco <think>."
    )

print(texto)


## Dia 5.7 — rembg/u2net local

In [ ]:

from pathlib import Path
from rembg import new_session

if "REMBG_MODEL_PATH" not in globals():
    raise RuntimeError(
        "REMBG_MODEL_PATH não foi definido. "
        "Execute primeiro a célula de bootstrap do u2net."
    )

modelo_onnx = Path(REMBG_MODEL_PATH)

if not modelo_onnx.is_file():
    raise RuntimeError(
        "Modelo rembg/u2net não existe no caminho resolvido: "
        + str(modelo_onnx)
    )

sessao = new_session(REMBG_MODEL)

print("[OK] rembg local:", REMBG_MODEL)
print("Modelo:", modelo_onnx)

del sessao


## Dia 5.8 — Avaliação dos 10 casos

In [ ]:
usar_dia("dia4")
import equipe as equipe_eval

from langchain_core.messages import HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver

_EVAL_CONN = None
_EVAL_GRAPH = None

def preparar_avaliacao():
    global _EVAL_CONN
    global _EVAL_GRAPH

    if _EVAL_CONN is not None:
        try:
            _EVAL_CONN.execute("SELECT 1")
            return _EVAL_GRAPH
        except Exception:
            try:
                _EVAL_CONN.close()
            except Exception:
                pass

    _EVAL_CONN = sqlite3.connect(
        str(
            N
            / "equipe_avaliacao_vscode_offline.db"
        ),
        check_same_thread=False,
    )

    saver = SqliteSaver(_EVAL_CONN)

    _EVAL_GRAPH = (
        equipe_eval
        .construir_equipe()
        .compile(checkpointer=saver)
    )

    return _EVAL_GRAPH

def fechar_avaliacao():
    global _EVAL_CONN
    global _EVAL_GRAPH

    if _EVAL_CONN is not None:
        _EVAL_CONN.close()

    _EVAL_CONN = None
    _EVAL_GRAPH = None

def responder_avaliacao(pergunta):
    g = preparar_avaliacao()

    entrada = {
        "messages": [HumanMessage(pergunta)],
        "pergunta": pergunta,
        "proximo": "",
        "instrucao": pergunta,
        "achados": [],
        "rascunho": "",
        "veredito": "",
        "rodadas": 0,
    }

    final = g.invoke(
        entrada,
        {
            "configurable": {
                "thread_id": f"eval-{uuid.uuid4()}"
            },
            "recursion_limit": 40,
        },
    )

    texto = (
        final.get("rascunho")
        or final["messages"][-1].content
        or ""
    )

    fontes = sorted(
        set(
            re.findall(
                r"\[fonte:\s*([^\]]+)\]",
                texto,
                flags=re.I,
            )
        )
    )

    return texto, fontes, 0

In [ ]:
RODAR_AVALIACAO_COMPLETA = False

usar_dia("dia5")
import avaliacao as avaliacao5

casos = avaliacao5.carregar_casos(
    N / "avaliacao" / "casos.jsonl"
)

print("Casos:", len(casos))

try:
    if RODAR_AVALIACAO_COMPLETA:
        resultados = avaliacao5.rodar(
            responder_avaliacao,
            casos,
        )
        print(
            avaliacao5.tabela(resultados)
        )
    else:
        print(
            "Avaliação completa desabilitada."
        )
finally:
    fechar_avaliacao()

# Finalização — validação final dos recursos locais

In [ ]:

raw_final, ollama_final = modelos_ollama_instalados()

ollama_faltando_final = [
    m
    for m in OLLAMA_OBRIGATORIOS
    if canon(m) not in ollama_final
]

hf_faltando_final = [
    repo_id
    for chave, repo_id in HF.items()
    if not resolver_hf_local(repo_id)
]

u2net_final = Path(
    REMBG_MODEL_PATH
) if "REMBG_MODEL_PATH" in globals() else None

if ollama_faltando_final:
    raise RuntimeError(
        "Validação final falhou para Ollama:\n"
        + "\n".join(ollama_faltando_final)
    )

if hf_faltando_final:
    raise RuntimeError(
        "Validação final falhou para Hugging Face:\n"
        + "\n".join(hf_faltando_final)
    )

if u2net_final is None or not u2net_final.is_file():
    raise RuntimeError(
        "Validação final falhou para rembg/u2net: "
        + str(u2net_final)
    )

print("Repositório:", R)
print("NEXUS:", N)
print("Ollama:", OLLAMA_BASE_URL)
print("HF cache:", HF_HUB_LOCAL)
print("u2net:", u2net_final)
print("\n[OK] TODOS OS MODELOS EXIGIDOS EXISTEM LOCALMENTE.")
